In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import cv2

def np_CountUpContinuingOnes(b_arr):
    left = np.arange(len(b_arr))
    left[b_arr > 0] = 0
    left = np.maximum.accumulate(left)
    rev_arr = b_arr[::-1]
    right = np.arange(len(rev_arr))
    right[rev_arr > 0] = 0
    right = np.maximum.accumulate(right)
    right = len(rev_arr) - 1 - right[::-1]
    return right - left - 1

def ExtractBreast(img):
    img_copy = img.copy()
    img = np.where(img <= 20, 0, img)
    height, _ = img.shape
    y_a = height // 2 + int(height * 0.4)
    y_b = height // 2 - int(height * 0.4)
    b_arr = img[y_b:y_a].std(axis=0) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    col_ind = np.where(continuing_ones == continuing_ones.max())[0]
    img = img[:, col_ind]
    _, width = img.shape
    x_a = width // 2 + int(width * 0.4)
    x_b = width // 2 - int(width * 0.4)
    b_arr = img[:, x_b:x_a].std(axis=1) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    row_ind = np.where(continuing_ones == continuing_ones.max())[0]
    return img_copy[row_ind][:, col_ind]

def process_data(source_folder, target_folder):
    # 读取数据划分CSV文件
    split_csv_path = "../classification_data/classification_split.csv"
    split_df = pd.read_csv(split_csv_path)
    # 只保留KAU-BCMD数据集
    split_df = split_df[split_df['dataset'] == 'KAU-BCMD']
    
    categories = ['Birad1', 'Birad3', 'Birad4', 'Birad5']
    saved_files = []

    # 创建目标文件夹结构
    for split in ['Train', 'Eval', 'Test']:
        os.makedirs(os.path.join(target_folder, split), exist_ok=True)

    for category in categories:
        category_folder = os.path.join(source_folder, category)
        
        # 遍历类别文件夹中的所有jpg文件
        for root, dirs, files in os.walk(category_folder):
            for file in files:
                if file.endswith('.jpg') and not file.startswith('._'):
                    file_name = os.path.splitext(file)[0].replace(' ', '')
                    relative_path = os.path.relpath(os.path.join(root, file), source_folder)
                    
                    # 查找对应的data_split
                    split_info = split_df[split_df['data_name'] == file_name]
                    if split_info.empty:
                        print(f"No split info found for {file_name}")
                        continue
                    
                    data_split = split_info['data_split'].values[0]
                    
                    # 处理并保存文件
                    save_image_and_info(
                        relative_path, 
                        source_folder, 
                        os.path.join(target_folder, data_split), 
                        category, 
                        saved_files
                    )
                    
    print("Data processing complete.")

def save_image_and_info(file_path, source_folder, target_folder, category, saved_files):
    file_name = os.path.splitext(os.path.basename(file_path))[0].replace(' ', '')
    target_folder_path = os.path.join(target_folder, file_name)
    os.makedirs(target_folder_path, exist_ok=True)
    target_img_path = os.path.join(target_folder_path, 'img.jpg')
    target_info_path = os.path.join(target_folder_path, 'info_dict.npy')

    # 处理并保存图片
    image = cv2.imread(os.path.join(source_folder, file_path), cv2.IMREAD_GRAYSCALE)
    processed_image = ExtractBreast(image)
    cv2.imwrite(target_img_path, processed_image)
    saved_files.append(file_path)

    # 创建并保存元数据
    if file_name in ['2018_BC005421_CC_R', '2018_BC005421_MLO_R', '2018_BC0022482_CC_R', '2018_BC0022482_MLO_R']:
        info_dict = {'Bi-Rads': "Bi-Rads 5"}
    else:
        info_dict = {'Bi-Rads': f"Bi-Rads {category[-1]}"}
    np.save(target_info_path, info_dict)
    print(f"{target_folder_path} has been saved")

if __name__ == "__main__":
    source_folder = "/Volumes/KAU-BCMD/archive"
    target_folder = "../classification_data/KAU-BCMD"
    
    process_data(source_folder, target_folder)